# AgroPilot AI — Multi-class Crop Leaf Model (Starter Training Notebook)

This notebook trains a real image-classification model with transfer learning on the public **PlantVillage** dataset, evaluates it on a held-out test split, and exports a TensorFlow.js model for the AgroPilot website.

**Coverage limit:** the PlantVillage starter dataset contains 38 leaf-condition classes across about 14 crop species; it does **not** cover every crop, every disease, or all real field conditions. It is not a universal all-crop diagnostic model. Add verified, correctly labelled local images for wheat, rice, mustard, chickpea, cotton and other missing crops before claiming support for them.

Run in Google Colab with a GPU runtime if available. Training may take time. The measured test metrics shown at the end are the results from your run; do not copy or invent accuracy values.

In [ ]:
# Install dependencies before importing TensorFlow.
!pip -q install tensorflow-datasets tensorflowjs

In [ ]:
import os, json, shutil, pathlib, numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

## 1. Download and inspect the dataset

The notebook downloads PlantVillage through TensorFlow Datasets. The labels are the dataset's official class names. Review the dataset terms and citation requirements before using this in a public/commercial product.

In [ ]:
(ds_all, info) = tfds.load(
    'plant_village',
    split='train',
    with_info=True,
    as_supervised=True,
    shuffle_files=True
)
class_names = info.features['label'].names
num_classes = len(class_names)
num_examples = info.splits['train'].num_examples
print('Images:', num_examples)
print('Classes:', num_classes)
for name in class_names:
    print('-', name)

## 2. Make train/validation/test splits

The held-out test portion is not used for gradient updates. This is a random image split, not a field/farm-separated split, so real-world performance can be lower than the reported test score.

In [ ]:
shuffled = ds_all.shuffle(min(num_examples, 8192), seed=SEED, reshuffle_each_iteration=False)
n_train = int(num_examples * 0.80)
n_val = int(num_examples * 0.10)
train_raw = shuffled.take(n_train)
val_raw = shuffled.skip(n_train).take(n_val)
test_raw = shuffled.skip(n_train + n_val)
print(f'Train: {n_train} | Validation: {n_val} | Test: {num_examples - n_train - n_val}')

def prepare(image, label, training=False):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)  # keep pixel range 0..255; model rescales internally
    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=12.0)
        image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

train_ds = train_raw.map(lambda x, y: prepare(x, y, True), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_raw.map(lambda x, y: prepare(x, y, False), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(lambda x, y: prepare(x, y, False), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## 3. Build a transfer-learning model

MobileNetV2 starts with ImageNet pretrained features. The final classification head is trained for PlantVillage's classes. This is a real ML training process, but it does not automatically make the model reliable on phone photos taken in fields.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
)
base_model.trainable = False
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='leaf_rgb_0_255')
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name='mobilenet_normalize')(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='class_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='agropilot_crop_doctor')
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train the classifier head

Start with a few epochs. You can increase epochs if the validation results are still improving and you have enough runtime.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('/content/agropilot_best.keras', monitor='val_accuracy', save_best_only=True)
]
history_head = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

## 5. Optional fine-tuning

Fine-tuning may improve the starter model but can also overfit. Keep Batch Normalization layers frozen and use a small learning rate.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_finetune = model.fit(train_ds, validation_data=val_ds, epochs=3, callbacks=callbacks)

## 6. Evaluate on the held-out test split

These numbers will be produced by your actual run. Check per-class performance as well as overall accuracy; some classes may be much harder than others.

In [ ]:
# Load best saved checkpoint when available.
checkpoint_path = '/content/agropilot_best.keras'
if os.path.exists(checkpoint_path):
    model = tf.keras.models.load_model(checkpoint_path)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f'HELD-OUT TEST LOSS: {test_loss:.4f}')
print(f'HELD-OUT TEST ACCURACY: {test_accuracy:.4%}')

## 7. Export a TensorFlow.js model for AgroPilot

The exported folder must contain `model.json`, one or more `*.bin` weight shards, and `labels.json`. Download the ZIP and extract its contents into the project's `models/crop-doctor/` folder.

In [ ]:
import tensorflowjs as tfjs
EXPORT_DIR = '/content/crop-doctor'
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
tfjs.converters.save_keras_model(model, EXPORT_DIR)
with open(os.path.join(EXPORT_DIR, 'labels.json'), 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)
zip_path = shutil.make_archive('/content/agropilot-crop-doctor-model', 'zip', EXPORT_DIR)
print('Export folder:', EXPORT_DIR)
print('Download this file:', zip_path)
print('Files:', os.listdir(EXPORT_DIR))

In [ ]:
from google.colab import files
files.download('/content/agropilot-crop-doctor-model.zip')

## 8. Before using this in the hackathon

- Copy `model.json`, every `.bin` shard, and `labels.json` into `agriculture-project-updated/models/crop-doctor/`.
- Run the website through VS Code Live Server, then refresh. Do not open the HTML as `file://` because model fetches may be blocked.
- The current UI intentionally blocks image predictions for selected crops absent from this model's classes (for example wheat, rice, mustard, chickpea and cotton). Expand the dataset and retrain before enabling them.
- Test with unseen images from different farms, lighting, phones, backgrounds and growth stages. Random splits from PlantVillage are not enough to validate field reliability.
- A softmax score is not a calibrated probability. Never recommend pesticide/fertilizer doses from image output alone.